# Topic: Full Fine-Tuning vs. PEFT (Parameter-Efficient Fine-Tuning)

## Definition (30-second explanation)
Think of a massive LLM like a fully constructed skyscraper. **Full Fine-Tuning** is like tearing out and rebuilding every single floor just to change the interior design—it's incredibly expensive and requires massive construction equipment. **PEFT** is like leaving the skyscraper exactly as it is, but putting up temporary, lightweight drywall in a few rooms. You freeze the original foundation and only add a tiny, highly efficient layer of new knowledge.

## Why Interviewers Ask This
Compute is the most expensive resource in GenAI. Interviewers need to know if you are a "budget-aware" engineer. If you suggest Full Fine-Tuning for a simple classification task, it signals you don't understand hardware constraints and might accidentally request $50,000 worth of A100 GPUs when a $500 consumer GPU would have sufficed.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** VRAM (GPU Memory). When training, you don't just store the model's weights. You have to store the Gradients (how much to change the weights) and the Optimizer States (Adam optimizer keeps a running average of gradients, costing 2x the memory of the weights themselves). 
*   **The Mechanism:** PEFT methods (like LoRA - Low-Rank Adaptation) **freeze** the base model's weights. They inject a tiny number of new, trainable parameters (often < 1% of the total model size). During backpropagation, gradients and optimizer states are *only* calculated and stored for this tiny 1% layer.
*   **The Trade-off:** PEFT might struggle slightly compared to Full Fine-Tuning if the new task requires a complete dismantling of the model's fundamental language understanding (e.g., teaching an English model to speak completely fluent Mandarin from scratch).

## When to Use
*   **PEFT:** Default choice for 95% of applied GenAI tasks (style transfer, Text-to-SQL, RAG alignment, tone matching). Mandatory when using consumer hardware.
*   **Full Fine-Tuning:** Pre-training a model from scratch, or adapting a model to a completely alien domain (e.g., training a base language model on raw genomic sequences or a completely new language).

## Advantages (PEFT)
*   Slashes VRAM requirements by 70-90%.
*   Drastically reduces training time and compute costs.
*   Prevents "Catastrophic Forgetting" (because the base knowledge is frozen, the model doesn't forget general facts while learning your specific task).
*   Storage efficient: You can store dozens of 100MB PEFT adapters for different tasks and swap them onto the same 15GB base model at runtime.

## Limitations (PEFT)
*   Can sometimes add slight inference latency if the adapter weights aren't permanently merged into the base model before deployment.
*   Hyperparameter tuning (like setting LoRA rank/alpha) can be finicky.

## Common Comparisons
*   **Full Fine-Tuning:** Updates 100% of weights. High memory, high cost, risk of catastrophic forgetting.
*   **PEFT (LoRA):** Updates < 1% of weights. Low memory, low cost, preserves base knowledge.

## Common Interview Traps
*   **The "PEFT is Faster" Trap:** Candidates often claim PEFT trains *faster* per step. It actually doesn't! The forward pass still has to go through the massive frozen model *plus* your new adapter. It saves **Memory (VRAM)**, not necessarily pure computation speed per step.
*   **Ignoring the Optimizer:** Candidates often forget that the Adam optimizer is the real memory killer in Full Fine-Tuning, not just the model weights.

## Important Formula (VRAM Memory Estimation)
Interviewers expect you to do back-of-the-napkin VRAM math.
*   **1 Parameter = 2 Bytes** (in 16-bit precision like fp16/bf16).
*   **Model Weights Memory:** Parameters × 2 Bytes. *(e.g., 8 Billion params = 16 GB)*
*   **Full Fine-Tuning Memory:** ~4x to 6x the Model Weights Memory. *(Weights + Gradients + Adam States). e.g., 16 GB × 4 = 64 GB VRAM.*
*   **PEFT Memory:** ~1x Model Weights Memory + a tiny fraction for the adapter. *(e.g., 16 GB + 1 GB = 17 GB VRAM).*

## 45-Second Interview Answer
"Full Fine-Tuning updates every weight in a model, which requires massive VRAM because you have to store gradients and optimizer states for billions of parameters. PEFT, or Parameter-Efficient Fine-Tuning, solves this by freezing the base model and only training a tiny fraction of new parameters, like LoRA adapters. I default to PEFT for downstream tasks because it cuts memory requirements by up to 90%, prevents catastrophic forgetting of the base model's knowledge, and allows me to train highly capable models on consumer hardware without sacrificing meaningful performance."

## Practice Questions:

### Q1: VRAM Estimation & The PEFT Solution
**Question:** Fine-tune Llama-3-8B (16-bit precision) on a 12GB RTX 4080. Walk me through the VRAM math for just the weights, Full Fine-Tuning, and explain how PEFT solves the bottleneck.

**Answer:**
1. **Model Weights:** At 16-bit precision (2 bytes per parameter), an 8 Billion parameter model requires roughly 16GB of VRAM just to sit in memory. 
2. **Full Fine-Tuning:** During training, we don't just store weights. We store gradients and optimizer states. The Adam optimizer is particularly heavy because it tracks two moments (average of past gradients and average of squared gradients) for every single parameter. This inflates the VRAM requirement to 4-6x the model size, putting Full Fine-Tuning at roughly 64GB to 96GB.
3. **The PEFT Solution:** Similar to feature extraction in traditional Deep Learning, PEFT freezes the massive 8B base model. We attach a small Low-Rank Adapter (LoRA) matrix. Because the base model is frozen, we *only* calculate and store gradients and Adam states for the tiny adapter network. 
4. **The Reality Check (QLoRA):** Standard PEFT would still require ~17GB (16GB base + 1GB adapter). To fit it on my 12GB GPU, I must use QLoRA: quantizing the frozen base weights to 4-bit precision (~4.5GB), leaving plenty of room for the adapter and training overhead.

**Interview Tips:**
*   **The Adam Flex:** Mentioning exactly *why* Adam takes up so much memory (tracking moment 1 and moment 2) is a massive green flag for interviewers.
*   **The 2-Byte Rule:** Always remember 1 Billion Params = ~2GB in 16-bit.